# Dataset exploration

In [1]:
import pandas as pd

X = pd.read_csv("../data/X_train_6ZIKlTY.csv")
Y = pd.read_csv("../data/y_train_lXj6X5y.csv")



In [2]:
eda = pd.concat([X, Y], axis=1)
print(eda.info())

<class 'pandas.DataFrame'>
RangeIndex: 55603 entries, 0 to 55602
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Index                  55603 non-null  int64  
 1   patient_id             55603 non-null  str    
 2   cohort                 55603 non-null  str    
 3   sexM                   55603 non-null  int64  
 4   gene                   37645 non-null  str    
 5   age_at_diagnosis       52773 non-null  float64
 6   age                    55603 non-null  float64
 7   ledd                   35010 non-null  float64
 8   time_since_intake_on   29663 non-null  float64
 9   time_since_intake_off  11775 non-null  float64
 10  on                     38944 non-null  float64
 11  off                    32196 non-null  float64
 12  Index                  55603 non-null  int64  
 13  target                 55603 non-null  float64
dtypes: float64(8), int64(3), str(3)
memory usage: 5.9 MB
None


In [4]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils import get_tags
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion, make_pipeline
from sklearn.model_selection import GroupShuffleSplit, cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor

# Shared encoding; identifiers are kept in X for patient-grouped splits.
encoder = ColumnTransformer(
    [("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
      ["cohort", "gene"])],
    remainder="passthrough", verbose_feature_names_out=False,
).set_output(transform="pandas")

preprocessor = Pipeline([
    ("encoding", encoder),
    ("remove_ids", FunctionTransformer(
        lambda df: df.drop(columns=["Index", "patient_id"])
    )),
])

# Cohort medians are learned on training rows only, with global fallback.
def cohorts(df):
    encoded = df.filter(regex="^cohort_")
    return encoded.idxmax(axis=1).where(encoded.max(axis=1) > 0, "unknown")

class CohortMedian(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        self.medians_ = X.median()
        self.by_cohort_ = X.groupby(cohorts(X)).median()
        return self

    def transform(self, X):
        fill = self.by_cohort_.reindex(cohorts(X))
        fill.index = X.index
        return X.fillna(fill).fillna(self.medians_)

# Use 10-year age bands; sparse groups fall back to cohort/global medians.
class CohortAgeSexMedian(CohortMedian):
    def __init__(self, age_width=10, min_count=10):
        self.age_width = age_width
        self.min_count = min_count

    def groups(self, X):
        return [cohorts(X), X["age"] // self.age_width, X["sexM"]]

    def fit(self, X, y=None):
        super().fit(X, y)
        grouped = X.groupby(self.groups(X))
        self.by_group_ = grouped.median().where(grouped.count() >= self.min_count)
        return self

    def transform(self, X):
        fill = self.by_group_.reindex(pd.MultiIndex.from_arrays(self.groups(X)))
        fill.index = X.index
        return super().transform(X.fillna(fill))

# Predict only these three missing features, using the current visit's data.
predictors = {
    "age_at_diagnosis": ["age", "sexM", "cohort_A", "cohort_B"],
    "on": ["off", "age", "age_at_diagnosis", "cohort_A", "cohort_B"],
    "off": ["on", "age", "age_at_diagnosis", "ledd",
            "time_since_intake_on", "cohort_A", "cohort_B"],
}

class PredictiveImputer(TransformerMixin, BaseEstimator):
    def __init__(self, predictors):
        self.predictors = predictors

    def fit(self, X, y=None):
        self.medians_, self.models_ = X.median(), {}
        for col, features in self.predictors.items():
            observed = X[col].notna()
            self.models_[col] = HistGradientBoostingRegressor(
                max_iter=50, max_leaf_nodes=15, random_state=42
            ).fit(X.loc[observed, features], X.loc[observed, col])
        return self

    def transform(self, X):
        filled = X.copy()
        for col, features in self.predictors.items():
            missing = X[col].isna()
            if missing.any():
                values = self.models_[col].predict(X.loc[missing, features])
                if col == "age_at_diagnosis":  # Diagnosis cannot follow this visit.
                    values = values.clip(max=X.loc[missing, "age"].to_numpy())
                filled.loc[missing, col] = values
        return filled.fillna(self.medians_)  # Other features keep median imputation.

# Add strategies here; each retains the original missing-value indicators.
strategies = {
    "median": SimpleImputer(strategy="median", add_indicator=True)
        .set_output(transform="pandas"),
    "cohort_age_sex_median": FeatureUnion([
        ("values", CohortAgeSexMedian()),
        ("missing", MissingIndicator(error_on_new=False)),
    ]),
    "multivariate_3": FeatureUnion([
        ("values", PredictiveImputer(predictors)),
        ("missing", MissingIndicator(error_on_new=False)),
    ]),
    "multivariate_4": FeatureUnion([
        ("values", PredictiveImputer({**predictors,
            "ledd": ["age", "age_at_diagnosis", "off", "on", "sexM", "cohort_A", "cohort_B"],
        })),
        ("missing", MissingIndicator(error_on_new=False)),
    ]),
    "native_nan": FeatureUnion([
        ("values", FunctionTransformer()),  # Keep numeric NaNs unchanged.
        ("missing", MissingIndicator(error_on_new=False)),
    ]),
}
models = {
    "LR": make_pipeline(StandardScaler(), LinearRegression()),
    #"SVM": make_pipeline(StandardScaler(), SVR()),
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42),
    "HistGB": HistGradientBoostingRegressor(random_state=42),
}

y = Y["target"]
cv = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
results = []
for strategy, transformer in strategies.items():
    for name, model in models.items():
        # Skip models that cannot handle the native NaNs.
        if strategy == "native_nan" and not get_tags(model).input_tags.allow_nan:
            continue
        pipeline = make_pipeline(
            clone(preprocessor), clone(transformer), clone(model)
        )
        scores = -cross_val_score(
            pipeline, X, y, groups=X["patient_id"], cv=cv,
            scoring="neg_mean_squared_error", error_score="raise",
        )
        results.append([strategy, name, scores.mean(), scores.std()])

results = pd.DataFrame(results, columns=["strategy", "model", "MSE", "std"])
print(results.sort_values("MSE").to_string(index=False))

# Median-baseline DataFrames remain available for exploration.
train_idx, val_idx = next(cv.split(X, y, groups=X["patient_id"]))
exploration_preprocessor = make_pipeline(
    clone(preprocessor), clone(strategies["median"])
)
X_train_processed = exploration_preprocessor.fit_transform(X.iloc[train_idx])
X_val_processed = exploration_preprocessor.transform(X.iloc[val_idx])


             strategy        model       MSE      std
       multivariate_4       HistGB 54.234350 0.721443
       multivariate_3       HistGB 54.394390 0.711938
               median       HistGB 55.038634 0.727533
           native_nan       HistGB 55.044310 0.781912
cohort_age_sex_median       HistGB 55.190336 0.785440
       multivariate_4      XGBoost 55.741237 0.580474
       multivariate_3      XGBoost 55.888449 0.697287
           native_nan      XGBoost 56.484614 0.733073
cohort_age_sex_median      XGBoost 56.534715 0.905780
               median      XGBoost 56.585474 0.747708
       multivariate_4 RandomForest 57.463077 0.878549
       multivariate_3           LR 57.620723 0.868185
       multivariate_4           LR 57.674792 0.858567
       multivariate_3 RandomForest 57.727718 0.938764
           native_nan RandomForest 58.523001 0.946406
               median RandomForest 58.789044 0.988394
cohort_age_sex_median RandomForest 58.859734 0.931391
               median       